# RAG with Chroma in Ollama



In [ ]:
%pip install -U langchain langchain-ollama langchain-chroma langchain-community pypdf --quiet

Import libraries

In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

Initializing the models: main LLM and embeddings model

In [2]:
llm = ChatOllama(model="gemma4:e4b", temperature=0)
embeddings = OllamaEmbeddings(model="embeddinggemma")

Ingest files from the data directory

In [ ]:
loader = PyPDFDirectoryLoader("./data")
docs = loader.load()
print(f"Loaded {len(docs)} documents") #Langchain counts pages not documents :)

Loaded 5 documents


Split the do documents into smaller chunks. 1000 chars per chunk is a good practice

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 14 chunks


Initialize ChromaDB with local storage for persistence and create a retriever

In [5]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
#Retrieve 3 chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

Create a prompt to link together the retrieved chunks and the user's prompt

In [6]:
template = """You are a specialized AI assistant. Use the provided documentation 
to answer the user's request. If the answer isn't in the context, be honest.

Context: {context}

Question: {question}

Helpful Answer:"""

custom_prompt = PromptTemplate(
    template=template, 
    input_variables=["context", "question"]
)

## Chain to create the answer only
First step runs two parallel processes to build a dictionary with the two keys the custom prompt is looking for. 

In [7]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | custom_prompt 
    | llm 
    | StrOutputParser()
)

In [8]:
print(type(rag_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


Test the chain

In [9]:
query = "Can you explain Concentrated Solar Power?"
response = rag_chain.invoke(query)

In [10]:
print(response)

Based on the provided documentation, I cannot explain Concentrated Solar Power. The documents focus on how solar power works generally, specifically detailing the process using **photovoltaic (PV) technology** and silicon cells.


If we ask a question that is not related to the context, the chain tells us accordingly

In [11]:
query = "What colour is the sky?"
response = rag_chain.invoke(query)
print(response)

The provided documentation discusses the process of photosynthesis, chlorophyll, and the raw materials involved (light energy, water, and carbon dioxide). It does not contain any information about the color of the sky.


## Chain that provides also information about the sources
If we want to provide information about where the context came from we can redefine the chain

In [12]:
map_chain = RunnableParallel({
    "context": retriever,
    "question": RunnablePassthrough()
    })
rag_chain_with_sources = map_chain |{
    "answer": custom_prompt | llm | StrOutputParser(),
    "sources": lambda x: x["context"]
    }


In [13]:
print(type(map_chain))
print(type(rag_chain_with_sources))

<class 'langchain_core.runnables.base.RunnableParallel'>
<class 'langchain_core.runnables.base.RunnableSequence'>


In [14]:
query = "Can you explain Concentrated Solar Power?"
response = rag_chain_with_sources.invoke(query)

In [16]:
from pprint import pprint
pprint(response)

{'answer': 'Based on the provided documentation, there is no information '
           'explaining Concentrated Solar Power. The documents focus on how '
           'solar power works generally, specifically detailing the process of '
           'Photovoltaic (PV) technology.',
 'sources': [Document(id='083cd09c-da41-4368-b7f6-d77362050b54', metadata={'page_label': '1', 'total_pages': 2, 'producer': 'Microsoft: Print To PDF', 'creationdate': '2026-05-02T13:59:32+10:00', 'source': 'data\\solar_power - Copy.pdf', 'creator': 'PyPDF', 'moddate': '2026-05-02T13:59:32+10:00', 'page': 0, 'author': '', 'title': '*solar_power.txt - Notepad'}, page_content="Harnessing the Star: How Solar Power Works\nSolar power represents one of humanity's most significant energy revolutions. It is \nthe technology that taps directly into the nearly limitless energy emanating from \nthe sun, transforming sunlight—a seemingly simple radiant source—into usable \nelectricity. Far from being a niche technology, sola

In [17]:
for doc in response["sources"]:
    print(f"File {doc.metadata["source"]}, Page {doc.metadata["page"]}")

File data\solar_power - Copy.pdf, Page 0
File data\solar_power.pdf, Page 0
File data\solar_power.pdf, Page 0
